# CoALA + ReAct Design Pattern
## Customer Support Agent

---

### What This Notebook Focuses On

**Fixed (CoALA structure):** Parse -> Parallel Retrieval -> Working Memory -> Decision -> Act -> Learn

**Variable (ReAct pattern):** Working Memory runs a **Thought -> Action -> Observation loop**.
The LLM thinks, acts on ONE tool, observes the real result, then thinks again.
Each observation can **change what happens next**.

### The key question ReAct answers:
> *"Given what I just observed, what should I do next?"* -- asked again after every tool call.

### Memory: Pinecone `coala-memory` (seeded in Notebook_5, also written by NB6)
Semantic and episodic memory retrieved from Pinecone inform the reasoning at every step.

## CoALA Control Flow -- ReAct Variant

```
USER MESSAGE
    |
    v
[PARSE]                 -- extract intent, order_id, sentiment
    |
    v
[PARALLEL RETRIEVAL]    -- Pinecone semantic + episodic
    |
    v
[WORKING MEMORY -- ReAct Loop]
    |
    |-- Thought: what do I know, what should I do?
    |-- Action: call ONE tool
    |-- Observation: real result from tool (CSV data, not simulated)
    |-- Thought: given this observation, what next?
    |-- ... repeat until Final Answer
    |
    v
[LEARNING PHASE]        -- write facts + episode back to Pinecone
```

**Contrast with Planning (Notebook_6):** Planning generates the full plan before any tool runs.
ReAct generates ONE action at a time, informed by what the previous action revealed.

**Crucial difference:** In the run below, the customer says their order is 'delayed'.
The agent fetches the order and discovers it is actually **Cancelled**.
A Planning agent would have generated the wrong plan (compensation for delay).
ReAct adapts -- it sees 'Cancelled' in the observation and changes course.

In [ ]:
import os, re, json, time
import pandas as pd
from typing import Optional
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
embedder = SentenceTransformer("paraphrase-MiniLM-L6-v2")
orders_df = pd.read_csv("order.csv")

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("coala-memory")
SEMANTIC_NS = "coala_semantic"
EPISODIC_NS = "coala_episodic"

print(f"Loaded {len(orders_df)} orders | Connected to Pinecone coala-memory")
print(orders_df.head(3))

In [ ]:
@tool
def fetch_order(order_id: int) -> str:
    """Fetch full order details from the database given an order ID."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    return json.dumps(result.iloc[0].to_dict(), default=str)

@tool
def check_shipping_status(order_id: int) -> str:
    """Get a human-readable shipping status message for an order."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    status = result.iloc[0]["status"]
    return {"Delivered": "Your order has been delivered.",
            "Shipped": "Your order is currently on the way.",
            "Processing": "Your order is still being prepared and has not shipped yet.",
            "Cancelled": "Your order has been cancelled."}.get(status, "Status unknown.")

@tool
def offer_compensation(order_id: int) -> str:
    """Apply a 10% discount to the customer's account as compensation for an order issue."""
    result = orders_df[orders_df["order_id"] == order_id]
    name = result.iloc[0]["user_name"] if not result.empty else "the customer"
    return f"10% discount successfully applied to {name}'s account."

@tool
def provide_order_info(order_id: int) -> str:
    """Provide detailed order information: product name, status, and order date."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    r = result.iloc[0]
    return f"Order #{r['order_id']} -- {r['product_name']}, Status: {r['status']}, Placed on: {r['date']}."

@tool
def escalate_to_human(order_id: Optional[int] = None) -> str:
    """Escalate a complex or unresolved issue to the human support team. order_id is optional."""
    suffix = f" for order {order_id}" if order_id else ""
    return f"The issue{suffix} has been escalated. A representative will contact you within 24 hours."

action_tools = [fetch_order, check_shipping_status, offer_compensation, provide_order_info, escalate_to_human]

In [ ]:
def retrieve_semantic(query: str, k: int = 3) -> list:
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = index.query(vector=qv, top_k=k, include_metadata=True, namespace=SEMANTIC_NS)
    hits = [m["metadata"]["text"] for m in res.get("matches", []) if m["score"] > 0.25]
    print(f"  [SemanticMemory] {len(hits)} facts retrieved")
    return hits

def retrieve_episodic(query: str, k: int = 2) -> list:
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = index.query(vector=qv, top_k=k, include_metadata=True, namespace=EPISODIC_NS)
    hits = [m["metadata"] for m in res.get("matches", []) if m["score"] > 0.25]
    print(f"  [EpisodicMemory] {len(hits)} past episodes retrieved")
    return hits

def learn_semantic(fact: str):
    vec = embedder.encode([fact], normalize_embeddings=True)[0].tolist()
    vid = f"fact_{int(time.time())}_{abs(hash(fact)) % 100000}"
    index.upsert(vectors=[(vid, vec, {"text": fact, "ts": int(time.time())})], namespace=SEMANTIC_NS)
    print(f"  [SemanticMemory.learn] {fact[:70]}")

def learn_episodic(episode: dict):
    summary = episode.get("summary", str(episode))
    vec = embedder.encode([summary], normalize_embeddings=True)[0].tolist()
    vid = f"ep_{int(time.time())}_{abs(hash(summary)) % 100000}"
    meta = {"summary": summary, "intent": episode.get("intent", ""),
            "outcome": episode.get("outcome", ""), "ts": int(time.time())}
    index.upsert(vectors=[(vid, vec, meta)], namespace=EPISODIC_NS)
    print(f"  [EpisodicMemory.learn] {summary[:70]}")

def parse_observation(message: str) -> dict:
    msg = message.lower()
    match = re.search(r'\b(50\d{2})\b', msg)
    order_id = int(match.group(1)) if match else None
    if any(w in msg for w in ["delay", "late", "not arrived", "not received", "not shipped"]):
        intent = "order_delay"
    elif any(w in msg for w in ["cancel", "cancellation", "cancelled"]):
        intent = "order_cancellation"
    elif any(w in msg for w in ["where", "status", "track", "when", "update", "details"]):
        intent = "order_status"
    else:
        intent = "general_enquiry"
    sentiment = "negative" if any(w in msg for w in ["unhappy", "angry", "frustrated", "unacceptable", "furious"]) else "neutral"
    print(f"  [PARSE] intent={intent}, order_id={order_id}, sentiment={sentiment}")
    return {"raw": message, "intent": intent, "order_id": order_id, "sentiment": sentiment}

## ReAct Working Memory -- Thought / Action / Observation Loop

The ReAct prompt template forces the LLM into a strict format:
```
Thought:      [what I know and what to do next]
Action:       [tool name]
Action Input: [tool arguments]
Observation:  [real result from tool -- filled automatically]
```
The `{agent_scratchpad}` variable accumulates all previous Thought/Action/Observation turns,
so each new Thought is informed by the full history of what has been observed so far.

Memory context (semantic + episodic) is injected as **partial variables** -- the agent sees
past resolutions and relevant facts in every Thought step.

In [ ]:
REACT_TEMPLATE = """You are a customer support agent using the ReAct (Reasoning + Acting) pattern.

Long-term memory retrieved from Pinecone:
Semantic facts: {semantic_context}
Past similar episodes: {episodic_context}

You have access to the following tools:
{tools}

Use this format STRICTLY for every step:

Thought: [what you know so far and what to do next -- re-evaluate after each Observation]
Action: [tool name from: {tool_names}]
Action Input: [tool arguments as a JSON dict]
Observation: [the tool result will appear here automatically]
... (repeat Thought / Action / Action Input / Observation as many times as needed)
Thought: I now have enough information to give a complete answer
Final Answer: [your complete, empathetic response to the customer]

IMPORTANT: After each Observation, re-read what you observed and update your plan.
If the observation contradicts your initial assumption, change course immediately.

Customer message: {input}
Thought:{agent_scratchpad}"""


def build_react_agent(semantic_ctx: list, episodic_ctx: list) -> AgentExecutor:
    """Build a ReAct AgentExecutor with retrieved memory injected as context."""
    semantic_block = "\n".join(f"- {f}" for f in semantic_ctx) if semantic_ctx else "None retrieved"
    episodic_block = "\n".join(e.get("summary", "") for e in episodic_ctx) if episodic_ctx else "None retrieved"

    prompt = PromptTemplate.from_template(REACT_TEMPLATE).partial(
        semantic_context=semantic_block,
        episodic_context=episodic_block,
    )
    agent = create_react_agent(llm=llm, tools=action_tools, prompt=prompt)
    return AgentExecutor(
        agent=agent,
        tools=action_tools,
        verbose=True,              # shows every Thought / Action / Observation
        handle_parsing_errors=True,
        max_iterations=6,
    )

## CoALA ReAct Agent

In [ ]:
class CoALAReActAgent:

    def handle(self, message: str) -> str:
        print(f"\nUSER: {message}")
        print("="*60)

        # 1. PARSE
        print("\n[PARSE]")
        parsed = parse_observation(message)

        # 2. PARALLEL RETRIEVAL from Pinecone
        print("\n[PARALLEL RETRIEVAL -- Pinecone coala-memory]")
        semantic_ctx = retrieve_semantic(message)
        episodic_ctx = retrieve_episodic(message)

        # 3. WORKING MEMORY -- ReAct loop (verbose=True shows every step)
        print("\n[WORKING MEMORY -- ReAct Thought/Action/Observation loop]")
        print("  Each Thought is informed by the previous Observation.")
        print("  The agent can change course at any step.")
        agent_executor = build_react_agent(semantic_ctx, episodic_ctx)
        result = agent_executor.invoke({"input": message})
        response = result["output"]

        # 4. LEARNING PHASE
        print("\n[LEARNING PHASE]")
        learn_semantic(f"ReAct resolved {parsed['intent']} adaptively using observation-driven reasoning")
        learn_episodic({
            "summary": f"{message[:60]} -> ReAct adaptive resolution, intent={parsed['intent']}",
            "intent": parsed["intent"],
            "outcome": "resolved",
        })

        print(f"\nFINAL RESPONSE:\n{response}")
        return response

## Run -- The Adaptation Scenario

The customer believes their order is delayed.
Order 5005 is actually **Cancelled** in the CSV.

**Planning (NB6) would fail here:** it would generate a 'delay compensation' plan upfront and
execute it even though the order is cancelled.

**ReAct adapts:** after observing `status: Cancelled` in the first tool result,
the agent changes its next action from compensation to escalation.
Watch the Thought steps change after each Observation.

In [ ]:
agent = CoALAReActAgent()

# The customer assumes delay, but order 5005 is Cancelled -- agent must adapt mid-loop
agent.handle("My order 5005 seems to be delayed. It was supposed to arrive last week and I am frustrated.")

In [ ]:
# Second scenario -- order 5003 is Processing, customer wants compensation
# Notice how the agent's thoughts differ from the first run after seeing a different status
agent.handle("My order 5003 still has not shipped. I want compensation.")

## Key Observations -- ReAct Pattern

**What to notice in the verbose output:**
- Every `Thought:` line shows the agent reasoning with what it knows so far
- Every `Observation:` line contains **real data from order.csv** -- not simulated
- After observing `status: Cancelled`, the agent changed its Action from compensation to escalation
- This mid-execution adaptation is **impossible in the Planning pattern**

**Cost of adaptation:** Multiple LLM calls (one per Thought step) vs. one call in Planning.
ReAct is smarter but more expensive.

---

| Aspect | Planning (NB6) | ReAct (this notebook) | Tool Use (NB8) |
|---|---|---|---|
| LLM calls per interaction | 1 (upfront) | N (one per Thought) | N (one per tool selection) |
| Can adapt mid-execution | No -- plan is fixed | Yes -- Observation drives next Thought | Yes |
| Execution by | Code (plan runner) | LangChain ReAct executor | LangChain tool executor |
| Best for | Known, stable workflows | Uncertain tasks needing discovery | Any grounded real-data task |